[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/herrrickshaw/global-stock-screener/blob/main/notebooks/04_portfolio_evaluator.ipynb)


# 📤 Portfolio Evaluator — upload holdings → Buy / Hold / Sell
Upload your **broker holdings / tax-P&L statement** (CSV or XLSX) *or* paste tickers.
Each holding gets a **BUY / HOLD / SELL** from a multi-factor model — valuation (P/E),
momentum (200-DMA, golden/death cross, returns), quality (ROE/ROCE/Piotroski/debt), and
**recovery odds**. Structurally broken names are flagged **IMPAIRED** (unlikely to ever
recover). Missing fundamentals are fetched live on demand. *Research only — not advice.*


In [ ]:
# Setup — clone the repo + pull the central LFS data (idempotent; safe to re-run)
!pip install -q lmdb pyarrow scikit-learn nsepython yfinance vaderSentiment openpyxl
!git lfs install
%cd /content
![ -d global-stock-screener ] || git clone --depth 1 https://github.com/herrrickshaw/global-stock-screener.git
%cd /content/global-stock-screener
!git lfs pull
import os; os.environ['BHAV_CACHE'] = '/content/cache'
import market_memory as mm, serving_layer as sl
mm.ensure_stm_seeds(verbose=False)   # derive 1y STM from the committed 5y LTM
sl.refresh(['IN', 'US'])             # build serving views used below


## Option A — upload a file (Colab)


In [ ]:
from google.colab import files            # Colab only
up = files.upload()                        # choose your holdings CSV/XLSX
path = next(iter(up))
import portfolio_evaluator as pe
tickers = pe.parse_file(path); print('parsed', len(tickers), 'tickers:', tickers[:20])
pe.evaluate(tickers, market='IN')          # set market='US' for US tickers


## Option B — just paste tickers


In [ ]:
import portfolio_evaluator as pe
pe.evaluate(pe.parse_tickers('RELIANCE, TCS, INFY, WHEELS, SANSERA'), market='IN')


The result table (also saved to `results/portfolio_review.csv`) shows, per stock: the
**Signal**, **P/E**, trailing return, how many **screens** it clears, and the **rationale**.
